### Overview

Cross-validation of word2vec-based machine learning models using SCAN-B HiSeq training set (80%)

#### Note: Use Scipy Version 1.12 to be able to import Gensim

In [1]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
import random

#### 1. Import and prepare data

In [2]:
train_data = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/SCANB_GSE202203/scanb_hiseq_train_test_sets/train_test_80_20/SCANB_HiSeq_pam50gene_tpm_counts_subtype_train_80.csv", 
                          header=0, index_col=0)

In [3]:
train_data.shape

(2204, 52)

In [4]:
X_train = train_data.iloc[:, 0:50]
Y_train = train_data.iloc[:, [50]]

# check if the indices of x and y training sets match
X_train.index.equals(Y_train.index)

True

In [5]:
# label encoding of Y_train
label_encoder = LabelEncoder()
Y_train['subtype'] = label_encoder.fit_transform(Y_train['subtype'])

# check class count before label encoding
print("Class count before label encoding")
print(train_data['subtype'].value_counts())

# check class count after label encoding
print("\nClass count after label encoding")
print(Y_train['subtype'].value_counts())

Class count before label encoding
subtype
LumA     1119
LumB      654
Basal     230
Her2      201
Name: count, dtype: int64

Class count after label encoding
subtype
2    1119
3     654
0     230
1     201
Name: count, dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_9456\4218632629.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Y_train['subtype'] = label_encoder.fit_transform(Y_train['subtype'])


#### 2. Cross-validation of model performance

In [6]:
# function to rank gene column names to create gene sentence
def get_top_genes(row):
    return row.sort_values(ascending=False).index.to_list()

In [7]:
# function to get average embedding
# need to average the word embeddings because it creates a fixed-length single vector representation for the entire sequence
# useful for machine learning models which needs a single input vector for each sample
# Word2vec creates (m,n) embedding vector for each sequence (m: number of words in the sequence, n: embedding vector size)
# each word has an embedding of shape (vector size,)
# the average of the embeddings have to be computed column wise to maintain the shape of embedding vector size
# averaging the mean row-wise is incorrect, because it reduces the embedding of an entire word into a single value.
def get_average_vector(sequence, model):
    vectors = [model.wv[word] for word in sequence if word in model.wv]
    # return a zero vector if no valid words
    if len(vectors) == 0:
        return np.zeros(model.vector_size)  
    return np.mean(vectors, axis=0)

In [8]:
# rank the gene names for all the samples
X_train_ranked_geneList = X_train.apply(lambda row: get_top_genes(row), axis=1)
X_train_ranked_geneList = pd.DataFrame(X_train_ranked_geneList , index=X_train.index, columns=['ranked_gene_list'])

In [9]:
seed = 42
np.random.seed(seed)
random.seed(seed)

# define stratified k fold
stratified_k_fold = StratifiedKFold(n_splits=5, shuffle = True, random_state = seed)

#### 2.1 Random forest

In [10]:
random.seed(seed)
np.random.seed(seed)

fold = 1
f1_pam50_rf = []
acc_pam50_rf = []
prec_pam50_rf = []
recall_pam50_rf = []
mcc_pam50_rf = []

for train_index, val_index in stratified_k_fold.split(Y_train, Y_train):
    print(f"Starting fold {fold}\n")
    
    # split the training set into train and validation sets for 5-fold cross validation
    print(f"Splitting training set into train and validation sets...")
    x_train, x_val = X_train_ranked_geneList.iloc[train_index], X_train_ranked_geneList.iloc[val_index]
    y_train, y_val = Y_train.iloc[train_index], Y_train.iloc[val_index]

    print(f"Shape of X_train and X_validation sets: {x_train.shape} and {x_val.shape}")
    print(f"Shape of y_train and y_validation sets: {y_train.shape} and {y_val.shape}")

    # train word2vec skip-gram
    w2v_model = Word2Vec(x_train['ranked_gene_list'], vector_size=800, sg=1, alpha=0.01, window=6, epochs=10, min_count=1,
                         seed=seed, workers=1)

    # Convert the sequence of each sample to averaged word embedding vector 
    x_train_vect = np.array([get_average_vector(seq, w2v_model) for seq in x_train['ranked_gene_list']])
    x_val_vect = np.array([get_average_vector(seq, w2v_model) for seq in x_val['ranked_gene_list']])

    print('Embedding vector sequence in training: ',len(x_train_vect[0]))
    print('Embedding vector sequence in validation: ',len(x_val_vect[0]))
    print('Embedding vector shape in training: ',x_train_vect.shape)
    print('Embedding vector shape in validation: ',x_val_vect.shape)

    # standardize the embedding vectors
    scaler = StandardScaler()
    x_train_vect_scaled = scaler.fit_transform(x_train_vect)
    x_val_vect_scaled = scaler.transform(x_val_vect)

    print('Embedding vector shape in training after scaling: ',x_train_vect_scaled.shape)
    print('Embedding vector shape in validation after scaling: ',x_val_vect_scaled.shape)

    # pca transformation
    pca_rf = PCA(n_components=14, random_state=seed)
    x_train_pca = pca_rf.fit_transform(x_train_vect_scaled)
    x_val_pca = pca_rf.transform(x_val_vect_scaled)

    print('Embedding vector shape in training after pca: ', x_train_pca.shape)
    print('Embedding vector shape in validation after pca: ', x_val_pca.shape)

    # build rf classifier
    rfc = RandomForestClassifier(n_estimators=100, min_samples_split=2, min_samples_leaf=1, max_depth=None, random_state=seed)

    # fit the rf classifier on the training fold sets
    rfc.fit(x_train_pca, y_train.values.ravel())

    # make predictions on the validation fold set
    y_pred = rfc.predict(x_val_pca)

    # calculate metric scores
    mcc = metrics.matthews_corrcoef(y_val.values.ravel(), y_pred)
    f1 = metrics.f1_score(y_val.values.ravel(), y_pred, average='macro')
    recall = metrics.recall_score(y_val.values.ravel(), y_pred, average='macro')
    precision = metrics.precision_score(y_val.values.ravel(), y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_val.values.ravel(), y_pred)
    
    print(metrics.classification_report(y_val.values.ravel(), y_pred, digits=4))

    mcc_pam50_rf.append(mcc)
    f1_pam50_rf.append(f1)
    recall_pam50_rf.append(recall)
    prec_pam50_rf.append(precision)
    acc_pam50_rf.append(accuracy)

    fold += 1


Starting fold 1

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets: (1763, 1) and (441, 1)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
Embedding vector sequence in training:  800
Embedding vector sequence in validation:  800
Embedding vector shape in training:  (1763, 800)
Embedding vector shape in validation:  (441, 800)
Embedding vector shape in training after scaling:  (1763, 800)
Embedding vector shape in validation after scaling:  (441, 800)
Embedding vector shape in training after pca:  (1763, 14)
Embedding vector shape in validation after pca:  (441, 14)
              precision    recall  f1-score   support

           0     1.0000    0.9565    0.9778        46
           1     1.0000    0.8000    0.8889        40
           2     0.9345    0.9554    0.9448       224
           3     0.8676    0.9008    0.8839       131

    accuracy                         0.9252       441
   macro avg     0.9505    0.9032    0.92

In [11]:
print(f'Mean Accuracy: {round(np.mean(acc_pam50_rf), 4)} \u00B1 {round(np.std(acc_pam50_rf), 4)}')
print(f'Mean Precision: {round(np.mean(prec_pam50_rf), 4)} \u00B1  {round(np.std(prec_pam50_rf), 4)}')
print(f'Mean Recall: {round(np.mean(recall_pam50_rf), 4)} \u00B1  {round(np.std(recall_pam50_rf), 4)}')
print(f'Mean F1: {round(np.mean(f1_pam50_rf), 4)} \u00B1  {round(np.std(f1_pam50_rf), 4)}')
print(f'Mean MCC: {round(np.mean(mcc_pam50_rf), 4)} \u00B1  {round(np.std(mcc_pam50_rf), 4)}')

Mean Accuracy: 0.9256 ± 0.0124
Mean Precision: 0.9335 ±  0.0239
Mean Recall: 0.9132 ±  0.0183
Mean F1: 0.9223 ±  0.0193
Mean MCC: 0.8824 ±  0.0195


#### 2.2 SVM 

In [12]:
random.seed(seed)
np.random.seed(seed)

fold = 1
f1_pam50_svm = []
acc_pam50_svm = []
prec_pam50_svm = []
recall_pam50_svm = []
mcc_pam50_svm = []

for train_index, val_index in stratified_k_fold.split(Y_train, Y_train):
    print(f"Starting fold {fold}\n")
    
    # split the training set into train and validation sets for 5-fold cross validation
    print(f"Splitting training set into train and validation sets...")
    x_train, x_val = X_train_ranked_geneList.iloc[train_index], X_train_ranked_geneList.iloc[val_index]
    y_train, y_val = Y_train.iloc[train_index], Y_train.iloc[val_index]

    print(f"Shape of X_train and X_validation sets: {x_train.shape} and {x_val.shape}")
    print(f"Shape of y_train and y_validation sets: {y_train.shape} and {y_val.shape}")

    # train word2vec skip-gram
    w2v_model = Word2Vec(x_train['ranked_gene_list'], vector_size=600, sg=1, alpha=0.01, window=5, epochs=5, min_count=1, 
                         seed=seed, workers=1)

    # Convert the sequence of each sample to averaged word embedding vector
    x_train_vect = np.array([get_average_vector(seq, w2v_model) for seq in x_train['ranked_gene_list']])
    x_val_vect = np.array([get_average_vector(seq, w2v_model) for seq in x_val['ranked_gene_list']])

    print('Embedding vector sequence in training: ',len(x_train_vect[0]))
    print('Embedding vector sequence in validation: ',len(x_val_vect[0]))
    print('Embedding vector shape in training: ',x_train_vect.shape)
    print('Embedding vector shape in validation: ',x_val_vect.shape)

    # standardize the embedding vectors
    scaler =StandardScaler()
    x_train_vect_scaled = scaler.fit_transform(x_train_vect)
    x_val_vect_scaled = scaler.transform(x_val_vect)

    print('Embedding vector shape in training after scaling: ',x_train_vect_scaled.shape)
    print('Embedding vector shape in validation after scaling: ',x_val_vect_scaled.shape)

    # pca transformation
    pca_svm = PCA(n_components=72, random_state=seed)
    x_train_pca = pca_svm.fit_transform(x_train_vect_scaled)
    x_val_pca = pca_svm.transform(x_val_vect_scaled)

    print('Embedding vector shape in training after pca: ', x_train_pca.shape)
    print('Embedding vector shape in validation after pca: ', x_val_pca.shape)
    
    # build svm classifier
    svm = SVC(C=1, gamma='scale', kernel='rbf', random_state=seed)

    # fit the svm classifier on the training fold sets
    svm.fit(x_train_pca, y_train.values.ravel())

    # make predictions on the validation fold set
    y_pred = svm.predict(x_val_pca)

    # calculate metric scores
    mcc = metrics.matthews_corrcoef(y_val.values.ravel(), y_pred)
    f1 = metrics.f1_score(y_val.values.ravel(), y_pred, average='macro')
    recall = metrics.recall_score(y_val.values.ravel(), y_pred, average='macro')
    precision = metrics.precision_score(y_val.values.ravel(), y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_val.values.ravel(), y_pred)
    
    print(metrics.classification_report(y_val.values.ravel(), y_pred, digits=4))

    mcc_pam50_svm.append(mcc)
    f1_pam50_svm.append(f1)
    recall_pam50_svm.append(recall)
    prec_pam50_svm.append(precision)
    acc_pam50_svm.append(accuracy)

    fold += 1


Starting fold 1

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets: (1763, 1) and (441, 1)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
Embedding vector sequence in training:  600
Embedding vector sequence in validation:  600
Embedding vector shape in training:  (1763, 600)
Embedding vector shape in validation:  (441, 600)
Embedding vector shape in training after scaling:  (1763, 600)
Embedding vector shape in validation after scaling:  (441, 600)
Embedding vector shape in training after pca:  (1763, 72)
Embedding vector shape in validation after pca:  (441, 72)
              precision    recall  f1-score   support

           0     1.0000    0.9565    0.9778        46
           1     1.0000    0.8750    0.9333        40
           2     0.9467    0.9509    0.9488       224
           3     0.8759    0.9160    0.8955       131

    accuracy                         0.9342       441
   macro avg     0.9556    0.9246    0.93

In [13]:
print(f'Mean Accuracy: {round(np.mean(acc_pam50_svm), 4)} \u00B1 {round(np.std(acc_pam50_svm), 4)}')
print(f'Mean Precision: {round(np.mean(prec_pam50_svm), 4)} \u00B1  {round(np.std(prec_pam50_svm), 4)}')
print(f'Mean Recall: {round(np.mean(recall_pam50_svm), 4)} \u00B1  {round(np.std(recall_pam50_svm), 4)}')
print(f'Mean F1: {round(np.mean(f1_pam50_svm), 4)} \u00B1  {round(np.std(f1_pam50_svm), 4)}')
print(f'Mean MCC: {round(np.mean(mcc_pam50_svm), 4)} \u00B1  {round(np.std(mcc_pam50_svm), 4)}')

Mean Accuracy: 0.9387 ± 0.0048
Mean Precision: 0.9484 ±  0.0093
Mean Recall: 0.9297 ±  0.0126
Mean F1: 0.9382 ±  0.0102
Mean MCC: 0.9035 ±  0.0078


#### 2.3 Logistic Regression

In [14]:
random.seed(seed)
np.random.seed(seed)

fold = 1
f1_pam50_lr = []
acc_pam50_lr = []
prec_pam50_lr = []
recall_pam50_lr = []
mcc_pam50_lr = []

for train_index, val_index in stratified_k_fold.split(Y_train, Y_train):
    print(f"Starting fold {fold}\n")
    
    # split the training set into train and validation sets for 5-fold cross validation
    print(f"Splitting training set into train and validation sets...")
    x_train, x_val = X_train_ranked_geneList.iloc[train_index], X_train_ranked_geneList.iloc[val_index]
    y_train, y_val = Y_train.iloc[train_index], Y_train.iloc[val_index]

    print(f"Shape of X_train and X_validation sets: {x_train.shape} and {x_val.shape}")
    print(f"Shape of y_train and y_validation sets: {y_train.shape} and {y_val.shape}")

    # train word2vec skip-gram
    w2v_model = Word2Vec(x_train['ranked_gene_list'], vector_size=900, sg=1, alpha=0.05, window=3, epochs=5, min_count=1,
                         seed=seed, workers=1)

    # Convert the sequence of each sample to averaged word embedding vector
    x_train_vect = np.array([get_average_vector(seq, w2v_model) for seq in x_train['ranked_gene_list']])
    x_val_vect = np.array([get_average_vector(seq, w2v_model) for seq in x_val['ranked_gene_list']])

    print('Embedding vector sequence in training: ',len(x_train_vect[0]))
    print('Embedding vector sequence in validation: ',len(x_val_vect[0]))
    print('Embedding vector shape in training: ',x_train_vect.shape)
    print('Embedding vector shape in validation: ',x_val_vect.shape)

    # standardize the embedding vectors
    scaler =StandardScaler()
    x_train_vect_scaled = scaler.fit_transform(x_train_vect)
    x_val_vect_scaled = scaler.transform(x_val_vect)

    print('Embedding vector shape in training after scaling: ',x_train_vect_scaled.shape)
    print('Embedding vector shape in validation after scaling: ',x_val_vect_scaled.shape)

    # pca transformation
    pca_lr = PCA(n_components=18, random_state=seed)
    x_train_pca = pca_lr.fit_transform(x_train_vect_scaled)
    x_val_pca = pca_lr.transform(x_val_vect_scaled)

    print('Embedding vector shape in training after pca: ', x_train_pca.shape)
    print('Embedding vector shape in validation after pca: ', x_val_pca.shape)

    # build lr classifier
    lr = LogisticRegression(C=0.05, solver='saga', penalty='l2', random_state=seed, max_iter=1000)

    # fit the lr classifier on the training fold sets
    lr.fit(x_train_pca, y_train.values.ravel())
    
    # make predictions on the validation fold set
    y_pred = lr.predict(x_val_pca)

    # calculate metric scores
    mcc = metrics.matthews_corrcoef(y_val.values.ravel(), y_pred)
    f1 = metrics.f1_score(y_val.values.ravel(), y_pred, average='macro')
    recall = metrics.recall_score(y_val.values.ravel(), y_pred, average='macro')
    precision = metrics.precision_score(y_val.values.ravel(), y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_val.values.ravel(), y_pred)
    
    print(metrics.classification_report(y_val.values.ravel(), y_pred, digits=4))

    mcc_pam50_lr.append(mcc)
    f1_pam50_lr.append(f1)
    recall_pam50_lr.append(recall)
    prec_pam50_lr.append(precision)
    acc_pam50_lr.append(accuracy)

    fold += 1


Starting fold 1

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets: (1763, 1) and (441, 1)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
Embedding vector sequence in training:  900
Embedding vector sequence in validation:  900
Embedding vector shape in training:  (1763, 900)
Embedding vector shape in validation:  (441, 900)
Embedding vector shape in training after scaling:  (1763, 900)
Embedding vector shape in validation after scaling:  (441, 900)
Embedding vector shape in training after pca:  (1763, 18)
Embedding vector shape in validation after pca:  (441, 18)
              precision    recall  f1-score   support

           0     0.9773    0.9348    0.9556        46
           1     0.9231    0.9000    0.9114        40
           2     0.9471    0.9598    0.9534       224
           3     0.9084    0.9084    0.9084       131

    accuracy                         0.9365       441
   macro avg     0.9390    0.9258    0.93

In [15]:
print(f'Mean Accuracy: {round(np.mean(acc_pam50_lr), 4)} \u00B1 {round(np.std(acc_pam50_lr), 4)}')
print(f'Mean Precision: {round(np.mean(prec_pam50_lr), 4)} \u00B1  {round(np.std(prec_pam50_lr), 4)}')
print(f'Mean Recall: {round(np.mean(recall_pam50_lr), 4)} \u00B1  {round(np.std(recall_pam50_lr), 4)}')
print(f'Mean F1: {round(np.mean(f1_pam50_lr), 4)} \u00B1  {round(np.std(f1_pam50_lr), 4)}')
print(f'Mean MCC: {round(np.mean(mcc_pam50_lr), 4)} \u00B1  {round(np.std(mcc_pam50_lr), 4)}')

Mean Accuracy: 0.9338 ± 0.0081
Mean Precision: 0.9399 ±  0.0105
Mean Recall: 0.9226 ±  0.0098
Mean F1: 0.9306 ±  0.0099
Mean MCC: 0.8954 ±  0.0125
